# 06. QLoRA, LoftQ, and QA-LoRA: Quantized Parameter-Efficient Fine-Tuning

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Dettmers et al. (2023, *QLoRA: Efficient Finetuning of Quantized LLMs*, NeurIPS); Li et al. (2023, *LoftQ*, ICLR 2024); Han et al. (2023, *QA-LoRA*, ICLR 2024).
- **Purpose:** Enable fine-tuning of 65B/70B LLMs on a single GPU using NormalFloat 4 (NF4), Double Quantization (DQ), alternating SVD residual initialization (LoftQ), and group-wise quantization adaptation (QA-LoRA).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. BitNet b1.58 Native Ternary Transformers](05_bitnet_b1_58_native_ternary_transformers.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. KV Cache Quantization KIVI and KVQuant](../05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use zero-initialized LoRA ($B = 0$) when immediate post-quantization inference is required without fine-tuning (use LoftQ alternating SVD initialization); do NOT merge standard LoRA adapters back into group-wise INT4 weights post-training (use QA-LoRA to align updates with the group quantization grid).

## Mathematical Foundations & Governing Equations

1. **NormalFloat 4 (NF4) Codebook**: $q_i = \frac{1}{2} ( Q_X(i/16) + Q_X((i+1)/16) )$, $X \sim \mathcal{N}(0, 1)$
2. **QLoRA Double Quantization (DQ)**: $c_1 = \max_{B_1}|W|$, $c_2 = \frac{\max_{B_2}(c_1)}{127.0}$, $q_2 = \text{clamp}(\lfloor c_1 / c_2 \rceil, -127, 127)$
3. **LoftQ SVD Residual Initialization**: $[U, S, V^\top] = \text{SVD}(\mathbf{W} - Q(\mathbf{W}_{\text{base}}))$, $\mathbf{A} = U \sqrt{S}$, $\mathbf{B} = \sqrt{S} V^\top$
4. **QA-LoRA Group Adaptation**: $\mathbf{A}_{\text{qa}} = \mathbf{A} \odot (\mathbf{s}_g \otimes \mathbf{1}_L)$, $\mathbf{Y} = \mathbf{X} \mathbf{W}_{\text{deq}} + \frac{\alpha}{r} (\mathbf{X} \mathbf{A}_{\text{qa}} \mathbf{B}) + \mathbf{b}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
NF4_LEVELS = torch.tensor([-1.0, -0.6961928, -0.52507305, -0.39491749, -0.28444138, -0.18477343, -0.09105004, 0.0, 0.0795803, 0.1609302, 0.2461123, 0.33791524, 0.44070983, 0.562617, 0.72295684, 1.0])

def nf4_quantize(x: torch.Tensor, block_size: int = 8):
    x_flat = x.reshape(-1, block_size)
    scale = torch.clamp(torch.max(torch.abs(x_flat), dim=-1, keepdim=True).values, min=1e-7)
    idx = torch.argmin(torch.abs((x_flat / scale).unsqueeze(-1) - NF4_LEVELS.to(x.device)), dim=-1)
    deq = NF4_LEVELS.to(x.device)[idx] * scale
    return deq.reshape(x.shape), idx.reshape(x.shape).float(), scale.squeeze(-1)

def double_quantize(c1: torch.Tensor, dq_block_size: int = 8):
    c1_flat = c1.reshape(-1, dq_block_size)
    c2 = torch.clamp(torch.max(c1_flat, dim=-1, keepdim=True).values / 127.0, min=1e-7)
    q2 = torch.clamp(torch.round(c1_flat / c2), -127.0, 127.0)
    return (q2 * c2).reshape(c1.shape), q2.reshape(c1.shape), c2.squeeze(-1)

def qlora_forward(x, w_deq, lora_a, lora_b, scaling, bias=None):
    y = x @ w_deq + (x @ lora_a @ lora_b) * scaling
    return (y + bias) if bias is not None else y

def qa_lora_forward(x, w_deq, lora_a, lora_b, group_scales, scaling, group_size=4, bias=None):
    a_qa = lora_a * group_scales.repeat_interleave(group_size, dim=0)
    y = x @ w_deq + (x @ a_qa @ lora_b) * scaling
    return ((y + bias) if bias is not None else y), a_qa

In [ ]:
# 1. Linear anchor parity
assert float(torch.max(torch.abs(weights["test_input"] @ weights["weight"] + weights["bias"] - weights["test_output"]))) < 1e-5

# 2. NF4 and Double Quantization parity
w_deq, idx, c1 = nf4_quantize(weights["w_orig"], block_size=8)
assert float(torch.max(torch.abs(w_deq - weights["w_nf4_deq"]))) < 1e-5
assert float(torch.max(torch.abs(idx - weights["nf4_indices"]))) < 1e-5
assert float(torch.max(torch.abs(c1 - weights["c1"]))) < 1e-5

c1_deq, q2, c2 = double_quantize(weights["c1"], dq_block_size=8)
assert float(torch.max(torch.abs(c1_deq - weights["c1_deq"]))) < 1e-5
assert float(torch.max(torch.abs(q2 - weights["q2"]))) < 1e-5
assert float(torch.max(torch.abs(c2 - weights["c2"]))) < 1e-5

# 3. QLoRA, LoftQ, and QA-LoRA forward parity
scaling = 2.0 / 4.0
out_qlora = qlora_forward(weights["x_raw"], weights["w_dq_deq"], weights["lora_a"], weights["lora_b"], scaling, weights["b_orig"])
assert float(torch.max(torch.abs(out_qlora - weights["out_qlora"]))) < 1e-5
out_loftq = qlora_forward(weights["x_raw"], weights["loftq_q_deq"], weights["loftq_a"], weights["loftq_b"], 1.0, weights["b_orig"])
assert float(torch.max(torch.abs(out_loftq - weights["out_loftq"]))) < 1e-5
out_qa, a_qa = qa_lora_forward(weights["x_raw"], weights["w_nf4_deq"], weights["lora_a"], weights["lora_b"], weights["group_scales"], scaling, 4, weights["b_orig"])
assert float(torch.max(torch.abs(out_qa - weights["out_qa_lora"]))) < 1e-5
assert float(torch.max(torch.abs(a_qa - weights["a_qa"]))) < 1e-5
print("All QLoRA, LoftQ, and QA-LoRA parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative Evaluation: QLoRA vs LoftQ SVD Residual Alignment vs QA-LoRA
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (float(torch.mean((y_fp32 - y_pred) ** 2)) + 1e-12))).item()

sqnr_qlora, sqnr_loftq, sqnr_qa = eval_sqnr(weights["out_qlora"]), eval_sqnr(weights["out_loftq"]), eval_sqnr(weights["out_qa_lora"])
print(f"Standard QLoRA (NF4+DQ) SQNR: {sqnr_qlora:.2f} dB")
print(f"LoftQ (SVD Res-Init) SQNR:    {sqnr_loftq:.2f} dB (+{sqnr_loftq - sqnr_qlora:.2f} dB gain)")
print(f"QA-LoRA (Group-Wise) SQNR:    {sqnr_qa:.2f} dB")
assert sqnr_loftq > sqnr_qlora + 1.0, "LoftQ SVD residual initialization must provide >1 dB SQNR gain!"
assert sqnr_qlora > 20.0, "QLoRA NF4+DQ forward pass must maintain >20 dB output fidelity!"

In [ ]:
# Latency Microbenchmark: Fused QLoRA Base GEMM + Rank-4 Adapter (1024x1024)
b, d_in, d_out, rank = 16, 1024, 1024, 4
x_bench, w_bench = torch.randn(b, d_in), torch.randn(d_in, d_out)
la_bench, lb_bench = torch.randn(d_in, rank) * 0.1, torch.randn(rank, d_out) * 0.1

t0 = time.perf_counter()
for _ in range(50):
    y_bench = x_bench @ w_bench + (x_bench @ la_bench @ lb_bench) * (2.0 / rank)
t_qlora = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused QLoRA Forward Pass ({b}x{d_in} @ {d_in}x{d_out}, r={rank}) Latency: {t_qlora:.3f} ms")
assert t_qlora < 25.0, "Fused QLoRA forward pass must complete in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **NF4 Quantile Optimality**: NormalFloat 4 partitions the standard normal bell curve into equal-probability bins, outperforming uniform INT4 by $+0.8\text{ dB}$ SQNR on zero-mean LLM weights.
2. **Double Quantization Memory Savings**: Quantizing block scales $c_1$ into 8-bit integers compresses quantization overhead from $0.5$ to $0.127$ bits/param, freeing over $3\text{ GB}$ on 65B/70B models.
3. **LoftQ Zero-Step Alignment**: Initializing low-rank adapter weights $A, B$ to match the top singular vectors of $W - Q(W)$ recovers up to $80\%$ of the initial quantization loss, eliminating initial perplexity degradation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_qlora_loftq_and_qa_lora_quantized_peft.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.md)
- **PyTorch 2.x Module:** [06_qlora_loftq_and_qa_lora_quantized_peft.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.py)
- **Functional JAX Module:** [06_qlora_loftq_and_qa_lora_quantized_peft.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.py)
- **Native Rust Candle Track:** [06_qlora_loftq_and_qa_lora_quantized_peft.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.rs)
- **Hardware Kernel (CPU / WGSL):** [06_qlora_loftq_and_qa_lora_quantized_peft.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.rs)
- **Golden SafeTensors:** [06_qlora_loftq_and_qa_lora_quantized_peft.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. BitNet b1.58 Native Ternary Transformers](05_bitnet_b1_58_native_ternary_transformers.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 01. KV Cache Quantization KIVI and KVQuant](../05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.ipynb) &rarr;